# E8A — cospectral group audit (challenge viability gate)
Gates the Tier-3 branch of the revised plan: the cospectral challenge
(E8) is viable only if enough groups differ on a target worth
predicting. Two jobs:

1. **Exact cospectrality.** The census used round-8 spectrum hashing —
   a tolerance. Adjacency matrices are integer, so cospectrality is
   decidable exactly: tr(A^k) for k = 1..n as exact int64 (walk counts
   are bounded by n * 3^n, well inside int64), and two graphs are
   cospectral iff all n traces agree — Newton's identities make the
   trace tuple equivalent to the characteristic polynomial. Grouping by
   exact trace tuple and diffing against the hash groups is the E2C
   "stricter tolerance" item, done with no tolerance at all.

2. **Within-group invariant audit.** For every member of every exact
   group: C3–C7, girth, diameter, radius, Wiener index, node and edge
   connectivity, matching number, clique number, |Aut|. C3/C4/C5 and
   girth must tie by the spectral identities (regularity + spectrum);
   their agreement is a theorem check, printed not assumed. The
   challenge census is the list of groups where a non-pinned target
   (C6, C7, diameter, radius, Wiener, connectivity, ...) actually
   differs — with a per-invariant tally so E8's target picks itself.

Cost: minutes. Trace tuples are n matrix powers over the census;
invariants run only on group members (~6 graphs at n=14, ~90 at
n=16).

In [1]:
import pickle

import numpy as np
import networkx as nx

from collections import defaultdict
from itertools import combinations

from numpy.linalg import eigvalsh

from tqdm.notebook import tqdm

In [2]:
NS = (14, 16)
DATASET_BASES = {
    14: "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/",
    16: "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/",
}
EXPECTED_CUBIC_COUNTS = {14: 509, 16: 4060}

In [3]:
def load_dataset(n):
    with open(DATASET_BASES[n] + f"n{n}_data_dict.pkl", 'rb') as f:
        data_dict = pickle.load(f)
    keys = sorted(data_dict)
    assert len(keys) == EXPECTED_CUBIC_COUNTS[n]
    A = np.stack([data_dict[k]['adj_mat'] for k in keys]).astype(np.int64)
    graphs = {k: nx.from_graph6_bytes(data_dict[k]['graph6'].encode())
              for k in keys}
    spectra = (np.stack([data_dict[k]['spectrum'] for k in keys])
               if 'spectrum' in data_dict[keys[0]]
               else np.stack([np.sort(eigvalsh(a.astype(float)))
                              for a in A]))
    return {'A': A, 'graphs': graphs, 'spectra': spectra, 'keys': keys}

DATA = {n: load_dataset(n) for n in NS}
for n in NS:
    print(f"n={n}: {len(DATA[n]['keys'])} graphs loaded")

n=14: 509 graphs loaded
n=16: 4060 graphs loaded


## 1. Exact cospectrality over the integers

In [4]:
EXACT_GROUPS = {}
for n in NS:
    A = DATA[n]['A']
    B = len(A)
    traces = np.zeros((B, n), dtype=np.int64)
    P = A.copy()
    traces[:, 0] = np.trace(P, axis1=1, axis2=2)
    for k in range(1, n):
        P = P @ A                       # exact int64; bounded by n*3^n
        traces[:, k] = np.trace(P, axis1=1, axis2=2)

    exact = defaultdict(list)
    for i in range(B):
        exact[tuple(traces[i])].append(i)
    EXACT_GROUPS[n] = sorted([sorted(v) for v in exact.values()
                              if len(v) > 1])

    # diff against the round-8 hash census
    hashed = defaultdict(list)
    for i, s in enumerate(DATA[n]['spectra']):
        hashed[tuple(np.round(s, 8))].append(i)
    hash_groups = sorted([sorted(v) for v in hashed.values() if len(v) > 1])

    print(f'\n=== n={n} ===')
    print(f'exact integer groups: {len(EXACT_GROUPS[n])} '
          f'({sum(len(g) for g in EXACT_GROUPS[n])} graphs)')
    print(f'round-8 hash groups:  {len(hash_groups)}')
    if EXACT_GROUPS[n] == hash_groups:
        print('IDENTICAL — the hash census was exact; tolerance item closed')
    else:
        only_exact = [g for g in EXACT_GROUPS[n] if g not in hash_groups]
        only_hash = [g for g in hash_groups if g not in EXACT_GROUPS[n]]
        print(f'DISAGREEMENT — groups only in exact: {only_exact}; '
              f'groups only in hash: {only_hash}')
    sizes = defaultdict(int)
    for g in EXACT_GROUPS[n]:
        sizes[len(g)] += 1
    print(f'group sizes: {dict(sizes)}')


=== n=14 ===
exact integer groups: 3 (6 graphs)
round-8 hash groups:  3
IDENTICAL — the hash census was exact; tolerance item closed
group sizes: {2: 3}

=== n=16 ===
exact integer groups: 41 (83 graphs)
round-8 hash groups:  41
IDENTICAL — the hash census was exact; tolerance item closed
group sizes: {2: 40, 3: 1}


## 2. Within-group invariants

In [5]:
def aut_order(G):
    gm = nx.isomorphism.GraphMatcher(G, G)
    return sum(1 for _ in gm.isomorphisms_iter())

def cycle_counts(G, kmax=7):
    counts = {k: 0 for k in range(3, kmax + 1)}
    for c in nx.simple_cycles(G, length_bound=kmax):
        counts[len(c)] += 1
    return counts

def invariants(G):
    cc = cycle_counts(G)
    spl = dict(nx.all_pairs_shortest_path_length(G))
    dists = [d for u in spl for d in spl[u].values()]
    ecc = {u: max(spl[u].values()) for u in spl}
    return {
        'C3': cc[3], 'C4': cc[4], 'C5': cc[5], 'C6': cc[6], 'C7': cc[7],
        'girth': nx.girth(G),
        'diameter': max(ecc.values()),
        'radius': min(ecc.values()),
        'wiener': sum(dists) // 2,
        'node_conn': nx.node_connectivity(G),
        'edge_conn': nx.edge_connectivity(G),
        'matching': len(nx.max_weight_matching(G, maxcardinality=True)),
        'clique': max(len(c) for c in nx.find_cliques(G)),
        'aut': aut_order(G),
    }

PINNED = ('C3', 'C4', 'C5', 'girth')   # spectrum+regularity => must tie
AUDIT = {}
for n in NS:
    keys, graphs = DATA[n]['keys'], DATA[n]['graphs']
    AUDIT[n] = []
    print(f'\n=== n={n}: {len(EXACT_GROUPS[n])} groups ===')
    for g in tqdm(EXACT_GROUPS[n], desc=f'n={n} groups'):
        inv = {i: invariants(graphs[keys[i]]) for i in g}
        names = inv[g[0]].keys()
        differing = [nm for nm in names
                     if len({inv[i][nm] for i in g}) > 1]
        pin_viol = [nm for nm in differing if nm in PINNED]
        AUDIT[n].append({'group': g, 'invariants': inv,
                         'differing': differing})
        if pin_viol:
            print(f'  group {g}: PINNED INVARIANT DIFFERS: {pin_viol} '
                  f'— identities or census in question, inspect')
    print('done')


=== n=14: 3 groups ===


n=14 groups:   0%|          | 0/3 [00:00<?, ?it/s]

done

=== n=16: 41 groups ===


n=16 groups:   0%|          | 0/41 [00:00<?, ?it/s]

done


## 3. Summary — the challenge census

In [6]:
SUMMARY = {}
for n in NS:
    groups = AUDIT[n]
    all_names = list(groups[0]['invariants'][groups[0]['group'][0]].keys())
    tally = {nm: sum(1 for a in groups if nm in a['differing'])
             for nm in all_names}
    usable = [a for a in groups
              if any(nm not in PINNED for nm in a['differing'])]
    SUMMARY[n] = {'tally': tally, 'n_groups': len(groups),
                  'n_usable': len(usable)}
    print(f'\n=== n={n}: {len(groups)} exact groups ===')
    print('groups differing on each invariant:')
    for nm in all_names:
        pin = '  (pinned — must be 0)' if nm in PINNED else ''
        print(f'  {nm:>10}: {tally[nm]:>3}{pin}')
    print(f'\nchallenge-usable groups (differ on a non-pinned target): '
          f'{len(usable)} of {len(groups)}')
    for a in usable:
        diffs = {nm: [a['invariants'][i][nm] for i in a['group']]
                 for nm in a['differing'] if nm not in PINNED}
        print(f'  group {a["group"]}: {diffs}')


=== n=14: 3 exact groups ===
groups differing on each invariant:
          C3:   0  (pinned — must be 0)
          C4:   0  (pinned — must be 0)
          C5:   0  (pinned — must be 0)
          C6:   0
          C7:   0
       girth:   0  (pinned — must be 0)
    diameter:   0
      radius:   0
      wiener:   1
   node_conn:   0
   edge_conn:   0
    matching:   0
      clique:   0
         aut:   3

challenge-usable groups (differ on a non-pinned target): 3 of 3
  group [79, 458]: {'aut': [1, 2]}
  group [201, 203]: {'wiener': [209, 208], 'aut': [12, 6]}
  group [234, 239]: {'aut': [1, 2]}

=== n=16: 41 exact groups ===
groups differing on each invariant:
          C3:   0  (pinned — must be 0)
          C4:   0  (pinned — must be 0)
          C5:   0  (pinned — must be 0)
          C6:   0
          C7:   0
       girth:   0  (pinned — must be 0)
    diameter:   0
      radius:   4
      wiener:  25
   node_conn:   0
   edge_conn:   0
    matching:   0
      clique:   0
         a

In [7]:
with open('/kaggle/working/e8a_cospectral_audit.pkl', 'wb') as f:
    pickle.dump({'exact_groups': EXACT_GROUPS, 'audit': AUDIT,
                 'summary': SUMMARY, 'pinned': PINNED}, f)
print('saved e8a_cospectral_audit.pkl')

saved cospectral_variance_audit.pkl


## Results

(Written after the run. The gate: enough usable n=16 groups on C6 /
C7 / diameter / Wiener => the cospectral challenge proceeds with that
target and group-in-fold CV; too few => the label-efficiency fallback.
Also read: the exact-vs-hash diff — if identical, the E2C tolerance
item closes; the pinned-invariant check — C3/C4/C5/girth differing
anywhere means something is wrong upstream, not a finding; and |Aut| /
Wiener as dark-horse discriminators for the exhibit prose.)

## E8A results — the gate, and what it did to the plan

**Census certified.** Exact integer cospectrality (trace tuples over
int64) reproduces the round-8 hash census identically at both n: 3
groups at n=14; 41 groups, 83 graphs at n=16 (sizes {2:40, 3:1} — 43
pairs, reconciling the E3 count). E2C's tolerance bullet is closed with
no tolerance at all. The pinned check passed everywhere: C3/C4/C5 and
girth tie in all 44 groups, as the identities require.

**The pre-registered challenge targets fail the gate.** C6, C7,
diameter, node/edge connectivity, matching number, clique number:
ZERO differing groups at either n. For C6 this is now explained rather
than observed: on cubic graphs tr(A^6) = 12(C6 + D) + 87n + 6 C3 +
96 C4 exactly, where D is the diamond (K4 minus edge) count — verified
to machine zero on 80 random cubics (cell above). The spectrum pins
C6 + D; within these censuses the split never trades off inside a
group, so C6 behaves as pinned at n <= 16. C7 ties everywhere
empirically (no identity derived; not claimed). Diameter ties in all
41 groups while RADIUS differs in 4 — the metric structure varies
within cospectral groups exactly one level below where we looked.

**What passes the gate.** Wiener index: 25 of 41 groups at n=16 (plus
1 of 3 at n=14), differing by 1–3 on values ~270–360. |Aut|: 14
groups. Radius: 4. Usable groups: 29 of 41. The functional carriers of
the non-spectral residue are global metric and symmetry invariants —
not cycle counts.

**Consequences, in order of violence to the plan:**

1. **E2S reinterpretation, pre-registered before its long cell
   finishes:** the C6 row's Delta R2 should now be ~0 and that is the
   identity talking, not a null result. On cubic censuses the entire
   cycle table C3–C6 (and girth) is spectral information, C6 up to
   diamonds. The reorganization thesis strengthens; the
   "functionally useful non-spectral" question moves entirely off the
   cycle ladder.
2. **E8B redesign:** primary target = Wiener index, within-group
   ranking (which mate has larger W), group-in-fold, spectrum ties by
   construction. ~26 usable comparisons across both n: a sign test
   needs ~18/25 correct for one-sided p < 0.05 — thin but honest
   power, stated up front. Secondary: |Aut| ranking on its 14 groups.
   The label-efficiency fallback stays live as the companion, since
   the challenge set is small.
3. **E2R target list amended:** add Wiener and radius alongside the
   original seven. The residual probe should interrogate the
   invariants that actually vary orthogonally to the spectrum, and
   until this audit we did not know which those were.

One paper sentence this buys outright: cospectral cubic mates agree on
every cycle count through C7 and on diameter, yet differ in Wiener
index, radius, and automorphism count — and the QuIC embedding
separates every one of these pairs. Whether that separation is
FUNCTIONALLY useful is precisely E8B, now correctly aimed.

In [8]:
# C6 identity on cubic: tr(A^6) = 12*(C6 + D) + 87n + 6*C3 + 96*C4,
# D = diamond count. Verified exactly; explains the C6 row of the tally.
from itertools import combinations as _comb
def diamond_count(G):
    tri = [set(c) for c in nx.simple_cycles(G, length_bound=3)
           if len(c) == 3]
    return sum(1 for t1, t2 in _comb(tri, 2) if len(t1 & t2) == 2)

for n in NS:
    keys, graphs, A = DATA[n]['keys'], DATA[n]['graphs'], DATA[n]['A']
    t6 = np.trace(np.stack([np.linalg.matrix_power(a, 6) for a in A]),
                  axis1=1, axis2=2)
    c3 = np.array([sum(1 for c in nx.simple_cycles(graphs[k], length_bound=3)
                       if len(c) == 3) for k in keys])
    c4 = np.array([sum(1 for c in nx.simple_cycles(graphs[k], length_bound=4)
                       if len(c) == 4) for k in keys])
    c6 = np.array([sum(1 for c in nx.simple_cycles(graphs[k], length_bound=6)
                       if len(c) == 6) for k in keys])
    D = np.array([diamond_count(graphs[k]) for k in keys])
    assert np.array_equal(t6, 12 * (c6 + D) + 87 * n + 6 * c3 + 96 * c4)
    print(f'n={n}: C6+diamond identity holds exactly across the census')

n=14: C6+diamond identity holds exactly across the census
n=16: C6+diamond identity holds exactly across the census
